# 9주차 · 무엇을 함께 권할까 — 연관규칙

**⏱ 60분** · 식료품점 영수증 9,835장

## 🎯 이번 주의 질문

마트는 계산대 쿠폰, 진열, 온라인 추천으로 고객에게 함께 살 상품을 권합니다. 무엇을 권할지는 영수증에서 **함께 산 품목**을 찾아 정합니다. "A를 산 고객은 B도 산다"는 꼴의 규칙을 **연관규칙**이라고 합니다.

> **어떤 품목을 산 고객에게 어떤 품목을 함께 권할까요?**

**이번 주에 할 일**
1. 영수증을 참·거짓 표로 바꾼다 (1절)
2. 규칙 하나의 지지도·신뢰도·향상도를 직접 계산한다 (2절)
3. `apriori`로 규칙을 모두 찾는다 (3절)
4. antecedent가 한 품목인 규칙을 읽고 권할 규칙을 정한다 (4~5절)

**데이터:** 한 식료품점의 30일 치 계산대 영수증 9,835장, 품목 범주 169개. Hahsler·Hornik·Reutterer(2006)가 공개하고 R `arules` 패키지에 실린 Groceries 자료입니다.

**반복할 패턴:** 참·거짓 표 만들기 → 지지도 기준으로 frequent itemset 찾기 → 신뢰도 기준으로 규칙 만들기 → 지지도·신뢰도·향상도로 읽기

배경이론: [9주차 이론](https://balab-pknu.github.io/bizanalytics/theory/week09.html)

| 셀 표시 | 할 일 |
|---|---|
| `# ✍️ 직접 입력` | 위에 보이는 코드를 직접 쳐서 실행합니다 |
| `# ▶ 그대로 실행하세요` | 준비·보조 코드입니다. 실행만 합니다 |
| `# 📝` | 과제입니다. 힌트를 보고 직접 씁니다 |

> **실습본입니다.** 먼저 메뉴 **파일 → Drive에 사본 저장**을 누르세요. 사본을 저장하지 않으면 입력한 코드가 사라집니다.
> 맨 위 준비 셀부터 순서대로 `Shift + Enter`로 실행합니다. ▶ 셀은 그대로 실행하고, ✍️ 셀은 위에 보여 준 코드를
> 빈 셀에 손으로 타이핑하고, 빈칸(`___`)은 채워서 실행합니다. 📝 과제는 힌트를 보고 스스로 풉니다.

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. 연관규칙 패키지 mlxtend를 설치하고 영수증 자료를 df로 읽습니다. Colab에 이미 있는 pandas·numpy는 그대로 둡니다.
%pip install -q "mlxtend==0.23.4"
import pandas as pd
pd.set_option("display.max_rows", 500)       # 표를 줄임표(...) 없이 끝까지 보여 줌
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", None)
from mlxtend.frequent_patterns import apriori, association_rules

import os
import sys
for _p in (".", "..", "../.."):
    if os.path.exists(os.path.join(_p, "balab.py")):
        sys.path.insert(0, _p); break
else:
    !wget -q https://raw.githubusercontent.com/BALAB-PKNU/bizanalytics/main/balab.py

from balab import load
df = load("groceries")
print("준비 완료!")

✅ `준비 완료!`가 나오면 됩니다.

---
## 1. 영수증 표

영수증 앞부분을 봅니다.

```python
df.head()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 **한 행 = 영수증 한 장**입니다. `items` 열에 그 영수증에서 산 품목이 쉼표로 나열되어 있습니다. 1번 영수증은 `citrus fruit`(감귤류)·`semi-finished bread`(반조리 빵)·`margarine`·`ready soups`(즉석 수프)를 함께 샀습니다. 품목은 `whole milk`(우유), `other vegetables`(기타 채소), `rolls/buns`(롤빵)처럼 169개 범주로 묶여 있고, 수량과 가격은 없습니다.

품목마다 열을 하나씩 만들어, 그 영수증에 있으면 참(`True`), 없으면 거짓(`False`)인 표로 바꿉니다. 규칙을 찾는 함수 `apriori`는 참·거짓 표를 받습니다. 쉼표로 나뉜 글자를 열로 펼치는 `str.get_dummies`는 이번 주에 익히지 않는 문법이라 그대로 실행합니다.

In [ ]:
# ▶ 그대로 실행하세요 — 영수증을 품목별 참·거짓 표로 바꿉니다
df = df["items"].str.get_dummies(sep=",").astype(bool)
df.shape

✅ `(9835, 169)`이면 맞습니다. 행은 영수증 9,835장, 열은 품목 169개입니다.

---
## 2. 지지도·신뢰도·향상도

참·거짓 열의 평균은 참의 비율입니다. 품목마다 그 품목이 든 영수증의 비율, 곧 **지지도(support)** 를 구해 큰 순서로 봅니다.

```python
df.mean().sort_values(ascending=False).head(10)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 `whole milk`(우유) 0.256, `other vegetables` 0.193, `rolls/buns` 0.184, `soda`(탄산음료) 0.174, `yogurt` 0.140 순입니다. 우유는 영수증 네 장 중 한 장꼴로 들어 있습니다.

규칙 **"요구르트를 산 영수증은 우유도 산다"** 를 계산해 봅니다. 화살표 앞(`yogurt`)을 **antecedent**, 뒤(`whole milk`)를 **consequent**라고 부릅니다. 먼저 두 품목을 함께 산 영수증을 표시하고 그 비율, 곧 이 규칙의 지지도를 구합니다. `&`는 두 조건을 모두 만족하는 행입니다.

```python
both = df["yogurt"] & df["whole milk"]
both.mean()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 0.056입니다. 영수증 9,835장 중 551장이 요구르트와 우유를 함께 샀습니다.

나머지 두 지표를 계산합니다.
- **신뢰도(confidence)** = antecedent를 산 영수증 가운데 consequent도 산 비율 = 0.056 ÷ 0.140 = 0.40. 요구르트를 산 영수증 1,372장 중 551장에 우유가 있습니다.
- **향상도(lift)** = 신뢰도 ÷ consequent의 지지도 = 0.40 ÷ 0.256 = 1.57. 요구르트를 산 영수증에서 우유가 나올 비율(0.40)이 아무 영수증에서 우유가 나올 비율(0.256)의 1.57배입니다. 향상도가 1이면 antecedent가 consequent와 관계없다는 뜻입니다.

```python
confidence = both.mean() / df["yogurt"].mean()
lift = confidence / df["whole milk"].mean()
confidence, lift
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ `(0.402..., 1.572...)`이면 맞습니다.

---
## 3. `apriori`로 규칙을 모두 찾는다

품목이 169개이면 만들 수 있는 규칙은 셀 수 없이 많습니다. `apriori`는 지지도가 기준 이상인 itemset, 곧 **frequent itemset**만 먼저 찾습니다. 기준 `min_support=0.01`은 영수증 약 98장 이상입니다. `use_colnames=True`는 itemset을 열 번호 대신 품목 이름으로 보여 줍니다.

```python
itemsets = apriori(df, min_support=0.01, use_colnames=True)
itemsets.shape
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 frequent itemset이 333개입니다.

frequent itemset에서 규칙을 만들고 신뢰도 0.4 이상만 남깁니다. 둘째 인자 `len(df)`는 전체 영수증 수입니다.

```python
rules = association_rules(itemsets, len(df), metric="confidence", min_threshold=0.4)
rules.shape
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 규칙 62개입니다. 표에는 열이 많고 itemset이 `frozenset({...})`으로 찍혀 읽기 어렵습니다. itemset을 글자로 바꾸고 다섯 열만 남겨 향상도 순으로 보여 주는 함수를 만듭니다.

In [ ]:
# ▶ 그대로 실행하세요 — 규칙 표를 읽기 쉽게 바꾸는 함수
def show_rules(rules):
    table = rules[["antecedents", "consequents", "support", "confidence", "lift"]].copy()
    table["antecedents"] = table["antecedents"].apply(lambda items: ", ".join(sorted(items)))
    table["consequents"] = table["consequents"].apply(lambda items: ", ".join(sorted(items)))
    return table.sort_values("lift", ascending=False, kind="stable").round(3)

향상도가 높은 규칙 10개를 봅니다. `antecedents`가 antecedent, `consequents`가 consequent입니다.

```python
show_rules(rules).head(10)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 1위가 `citrus fruit, root vegetables` → `other vegetables`(신뢰도 0.586, 향상도 3.03)이면 맞습니다.

💬 **결과 해설**
- 상위 10개 규칙의 consequent는 모두 `other vegetables`입니다. antecedent는 뿌리채소·과일·유제품 같은 신선식품 조합입니다.
- 지지도는 0.010~0.023으로, 영수증 약 98~228장에서 나온 규칙입니다.

규칙 62개 전체를 향상도 순으로 봅니다.

```python
show_rules(rules)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 표를 끝까지 내려 보면 consequent에는 `other vegetables`와 `whole milk` 두 품목만 나옵니다. 개수를 셉니다.

```python
rules["consequents"].value_counts()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 consequent는 `whole milk` 43개, `other vegetables` 19개뿐입니다. 두 품목은 영수증의 26%와 19%에 들어 있는, 가장 흔한 품목입니다. 원래 잘 팔리는 품목이 consequent에 오면 무엇을 산 영수증이든 신뢰도가 높게 나옵니다. 그래서 신뢰도와 함께 향상도를 봅니다.

---
## 4. 권할 규칙 고르기

권할 때 쓰기 쉬운 것은 antecedent가 한 품목인 규칙입니다. "이 품목을 산 고객에게 무엇을 권할까"에 바로 답합니다. `apply(len)`은 antecedent itemset의 품목 수입니다.

```python
show_rules(rules[rules["antecedents"].apply(len) == 1]).head(10)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 **결과 해설**
- 향상도 1위는 `onions` → `other vegetables`(신뢰도 0.459, 향상도 2.37)이지만 지지도는 0.014, 영수증 140장입니다.
- `root vegetables` → `other vegetables`는 신뢰도 0.435, 향상도 2.25이고, 지지도가 0.047로 영수증 466장에서 나왔습니다. 위 10개 가운데 근거 영수증이 가장 많습니다.
- `whole milk`가 consequent인 규칙은 신뢰도가 0.45~0.50으로 높지만 향상도는 1.76~1.95입니다. 우유는 원래 영수증의 26%에 들어 있어, 권하지 않아도 사는 고객이 많습니다.

---
## 5. 결정

> **뿌리채소(`root vegetables`)를 산 고객에게 기타 채소(`other vegetables`)를 권한다 (신뢰도 0.43, 향상도 2.25, 영수증 466장).**

- 향상도가 2를 넘는 규칙 가운데 근거 영수증이 가장 많습니다. `onions` 규칙은 향상도가 조금 높지만 영수증 140장에서 나왔습니다.
- 우유는 consequent로 가장 자주 나오지만, antecedent가 한 품목인 규칙의 향상도가 1.57~1.95로 기타 채소 규칙(2.08~2.37)보다 낮아 우선순위에서 뒤로 둡니다.
- 권한 효과는 권한 고객과 권하지 않은 고객을 나눠 비교해야 압니다(과제 3).

---
## 6. 실습 과제

### 📝 과제 1 · 규칙 하나 직접 계산

2절처럼 규칙 **{soda} ⇒ {whole milk}** 의 지지도·신뢰도·향상도를 계산하세요. 향상도는 1보다 큰가요?

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 2절의 두 셀에서 품목 이름만 바꿉니다. 마지막 줄에 지지도·신뢰도·향상도를 쉼표로 이어 적으면 셋을 함께 봅니다.


✏️ **나의 답:** (여기를 두 번 눌러 적습니다)

### 📝 과제 2 · 신뢰도 기준 높이기

3절에서 신뢰도 기준을 0.4에서 **0.5**로 높입니다. 규칙은 몇 개가 되고, 남은 규칙의 antecedent는 몇 품목인가요?

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 3절의 association_rules 한 줄에서 min_threshold만 바꾸고 shape를 봅니다. 규칙 표는 show_rules로 봅니다.


✏️ **나의 답:**

### 📝 과제 3 · 생각해 보기

뿌리채소를 산 고객에게 기타 채소 할인 쿠폰을 줬더니 기타 채소를 많이 샀습니다. 그 구매가 쿠폰 덕분인지 어떻게 확인할 수 있을까요? 1주차에 배운 방법과 연결해 적으세요.

✏️ **나의 답:**

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 지지도 | itemset을 산 영수증의 비율. {yogurt, whole milk} = 0.056 |
| 신뢰도 | antecedent를 산 영수증 중 consequent도 산 비율. 0.056 ÷ 0.140 = 0.40 |
| 향상도 | 신뢰도 ÷ consequent의 지지도. 0.40 ÷ 0.256 = 1.57. 1이면 관계없음 |
| `apriori` → `association_rules` | 지지도 기준으로 frequent itemset을 찾고, 신뢰도 기준으로 규칙을 만든다 |
| 신뢰도 읽기 | 흔한 품목(우유·기타 채소)이 consequent면 신뢰도가 저절로 높다 |
| 권할 규칙 | antecedent가 한 품목인 규칙을 신뢰도·향상도·지지도(영수증 수)로 함께 본다 |